# K-Means Experiments (Thesis-aligned)

Notebook terbagi menjadi bagian-bagian sesuai rencana tesis: impor library, penentuan K (Elbow + Silhouette), pelatihan model K-Means final, analisis cluster, dan jalankan eksperimen per-variant (baseline vs PCA).

In [ ]:
# Part 1 — Import libraries and configuration
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import gc
from tqdm import tqdm
import time

# ============================================================================
# CONFIGURATION - Specify your embedding file(s) directly
# ============================================================================

# Option 1: Single file (BGL or Thunderbird)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
    # Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
]

# Option 2: Multiple files (Combined dataset)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
# ]

# Option 3: PCA variants (smaller, faster)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
# ]

# Option 4: PCA128 for ultra-large datasets
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
# ]

RANDOM_STATE = 42
SAMPLE_FOR_METRICS = 200000  # sample size for silhouette if dataset is huge

print("📁 Input files configured:")
for f in INPUT_FILES:
    if f.exists():
        size_gb = f.stat().st_size / (1024**3)
        print(f"  ✓ {f.name} ({size_gb:.2f} GB)")
    else:

        print(f"  ❌ NOT FOUND: {f}")

In [ ]:
# Helper: Smart file loading functions with auto-dimension detection

def detect_embedding_dim(file_path: Path) -> int:
    """
    Auto-detect embedding dimension from filename pattern
    - *pca256* → 256 dims
    - *pca128* → 128 dims
    - default → 768 dims
    """
    filename = file_path.name.lower()
    if 'pca256' in filename:
        return 256
    elif 'pca128' in filename:
        return 128
    else:
        return 768

def infer_num_rows(path: Path, embedding_dim: int = None) -> int:
    """
    Infer number of rows for RAW memmap files
    Auto-detects dimension from filename if not provided
    """
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(path)
    size = path.stat().st_size
    return size // (embedding_dim * np.dtype(np.float32).itemsize)

def load_single_file_smart(file_path: Path, embedding_dim: int = None):
    """
    Smart loader: auto-detect .npy vs RAW memmap
    Returns (array, is_memmap, num_rows)
    
    Auto-detects dimension from filename if not provided:
    - *pca256* → 256 dims
    - *pca128* → 128 dims  
    - default → 768 dims
    """
    # Auto-detect dimension if not provided
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(file_path)
        print(f"   🔍 Auto-detected dimension: {embedding_dim} from filename")
    
    try:
        # Try standard .npy load first (works for small files)
        arr = np.load(file_path, mmap_mode='r')
        detected_dim = arr.shape[1]
        if detected_dim != embedding_dim:
            print(f"   ⚠️ Dimension mismatch! Expected {embedding_dim}, got {detected_dim} from .npy header")
            embedding_dim = detected_dim
        return arr, True, arr.shape[0]
    except Exception:
        # Fallback to RAW memmap (for ultra-large files from after_bert_pipeline)
        num_rows = infer_num_rows(file_path, embedding_dim)
        arr = np.memmap(
            file_path, 
            dtype=np.float32, 
            mode='r', 
            shape=(num_rows, embedding_dim)
        )
        print(f"   ⚠️ Loaded as RAW memmap: {num_rows:,} rows × {embedding_dim} dims")
        return arr, True, num_rows

def load_embeddings_from_files(files, force_copy=False):
    """
    Load embeddings from list of file paths with smart memory management
    
    Args:
        files: list of Path objects
        force_copy: force copy to RAM (use only for small datasets)
    
    Returns:
        Stacked array or memmap reference
    """
    if len(files) == 0:
        raise FileNotFoundError('No embedding files provided')
    
    # Verify all files exist
    for f in files:
        if not f.exists():
            raise FileNotFoundError(f'File not found: {f}')
    
    # Determine embedding dimension from first file (auto-detect)
    first_arr, _, _ = load_single_file_smart(files[0])
    embedding_dim = first_arr.shape[1]
    print(f"Embedding dimension: {embedding_dim}")
    
    # Single file case - return directly
    if len(files) == 1:
        print(f"Loading single file: {files[0].name}")
        if force_copy and not isinstance(first_arr, np.memmap):
            return np.array(first_arr)
        return first_arr
    
    # Multiple files - need to stack
    print(f"Loading {len(files)} files...")
    
    # Check total size to decide strategy
    total_samples = 0
    file_info = []
    for f in files:
        arr, is_mmap, n_rows = load_single_file_smart(f)
        file_info.append((f, arr, n_rows))
        total_samples += n_rows
        print(f"  - {f.name}: {n_rows:,} rows")
    
    total_size_gb = (total_samples * embedding_dim * 4) / (1024**3)
    print(f"\nTotal samples: {total_samples:,} ({total_size_gb:.2f} GB)")
    
    # Strategy 1: Small dataset (< 20GB) - load to RAM for speed
    if total_size_gb < 20 and force_copy:
        print("Strategy: Load all to RAM (fast clustering)")
        arrays = [arr[:] if isinstance(arr, np.memmap) else arr for _, arr, _ in file_info]
        return np.vstack(arrays)
    
    # Strategy 2: Large dataset - keep as memmap and use chunked processing
    # WARNING: vstack creates a copy! For ultra-large (100GB+), use single file or external merge
    if total_size_gb < 100:
        print("Strategy: Memory-mapped stacking (may use temp disk space)")
        arrays = [arr for _, arr, _ in file_info]
        return np.vstack(arrays)  # This creates memmap-backed array
    
    # Strategy 3: Ultra-large (>100GB) - CANNOT fit in memory
    print("⚠️ ULTRA LARGE DATASET DETECTED (>100GB)")
    print("⚠️ Stacking not recommended - use SINGLE file or process per-file!")
    raise MemoryError(
        f"Dataset too large ({total_size_gb:.1f}GB). "
        "Recommendations:\n"
        "1. Use single file at a time\n"
        "2. OR process each file separately\n"
        "3. OR use PCA variants (smaller size)"
    )

# Quick check configured files
print("\n" + "="*70)
print("FILE CHECK")
print("="*70)
for f in INPUT_FILES:
    if f.exists():
        size_gb = f.stat().st_size / (1024**3)
        detected_dim = detect_embedding_dim(f)
        print(f'✓ {f.name}')
        print(f'  Path: {f}')
        print(f'  Size: {size_gb:.2f} GB')
        print(f'  Auto-detected dimension: {detected_dim}')
    else:
        print(f'❌ NOT FOUND: {f}')
print("="*70)


## Test: File Type Detection & Size Analysis

Run this cell untuk analyze file types dan ukuran sebelum processing.

In [ ]:
# Diagnostic: Analyze configured files to determine loading strategy
print("🔍 Analyzing configured files for optimal loading strategy...\n")

total_size_gb = 0
file_details = []

for f in INPUT_FILES:
    if not f.exists():
        print(f"❌ File not found: {f}")
        continue
        
    size_gb = f.stat().st_size / (1024**3)
    total_size_gb += size_gb
    
    # Auto-detect dimension from filename
    auto_dim = detect_embedding_dim(f)
    
    # Try to detect file type
    try:
        test_arr = np.load(f, mmap_mode='r')
        file_type = "Standard .npy"
        shape = test_arr.shape
        actual_dim = test_arr.shape[1]
        del test_arr
        
        # Verify auto-detection matches
        if actual_dim != auto_dim:
            print(f"⚠️ Dimension mismatch for {f.name}:")
            print(f"   Auto-detected: {auto_dim}, Actual: {actual_dim}")
            print(f"   Using actual dimension from .npy header")
    except:
        # RAW memmap - use auto-detected dimension
        file_type = "RAW memmap"
        n_rows = infer_num_rows(f, embedding_dim=auto_dim)
        shape = (n_rows, auto_dim)
        actual_dim = auto_dim
    
    file_details.append({
        'name': f.name,
        'size_gb': size_gb,
        'type': file_type,
        'shape': shape,
        'dimension': actual_dim
    })

if len(file_details) == 0:
    print("⚠️ No valid files found! Check INPUT_FILES configuration.")
else:
    # Display results
    df = pd.DataFrame(file_details)
    print(df.to_string(index=False))
    
    print(f"\n{'='*60}")
    print(f"Total files: {len(file_details)}")
    print(f"Total size: {total_size_gb:.2f} GB")
    print(f"Total samples: {sum(d['shape'][0] for d in file_details):,}")
    
    # Recommend strategy
    print(f"\n📊 RECOMMENDED STRATEGY:")
    if total_size_gb < 20:
        print("✅ Load ALL to RAM (fast, <20GB)")
        print("   → Full dataset clustering with standard KMeans")
        print("   → Expected time: 15-20 minutes")
    elif total_size_gb < 100:
        print("⚠️  Memory-mapped processing (20-100GB)")
        print("   → Use MiniBatchKMeans")
        print("   → Expected time: 1-2 hours")
    else:
        print("🔥 ULTRA LARGE - Incremental processing (>100GB)")
        print("   → Automatic sampling + incremental fit")
        print("   → Expected time: 3-5 hours")
        print("   → Consider using PCA variants to reduce size")
    
    # Memory estimate
    print(f"\n💾 RAM requirement estimate:")
    print(f"   Full load: ~{total_size_gb:.1f} GB")
    print(f"   With KMeans overhead: ~{total_size_gb * 1.3:.1f} GB")
    print(f"   Recommended free RAM: ~{total_size_gb * 1.5:.1f} GB")


## Part 2 — Determine K (Elbow + Silhouette)
Rencana: gunakan Elbow (inertia) untuk range K, dan silhouette score (sample jika dataset besar). Jika dataset sangat besar gunakan `MiniBatchKMeans` untuk uji cepat.

### 🔧 Smart Loading for Mixed File Types

Notebook ini telah diupdate untuk handle **2 jenis file .npy**:

1. **Standard .npy** (BGL ~12GB): Menggunakan `np.load()` dengan mmap
2. **RAW memmap** (Thunderbird ~600GB): Menggunakan `np.memmap()` langsung

**Auto-Detection Strategy:**
- File < 20GB → Load ke RAM (fast clustering)
- File 20-100GB → Memory-mapped processing
- File > 100GB → Sampling + Incremental fitting

**Perubahan utama:**
- ✅ `load_single_file_smart()`: Auto-detect .npy vs RAW memmap
- ✅ `load_concat()`: Smart memory management berdasarkan ukuran total
- ✅ Part 2: Sampling strategy untuk K-selection pada dataset ultra-besar
- ✅ Part 3: Incremental fitting untuk Thunderbird-sized datasets


In [ ]:
# Part 2 — compute inertia and silhouette for K range
from sklearn.utils import resample

K_RANGE = range(2, 16)
inertia = []
sil_scores = []

# Load embeddings with smart strategy
print('Loading embeddings...')

# CRITICAL: For ultra-large datasets (Thunderbird 600GB), we CANNOT load all to RAM
# Check total size first
total_size_gb = sum(f.stat().st_size for f in INPUT_FILES if f.exists()) / (1024**3)
print(f'Total dataset size: {total_size_gb:.2f} GB')

# Auto-select strategy based on size
if total_size_gb > 100:
    print('⚠️ ULTRA LARGE DATASET - Using sampling strategy for K-selection')
    USE_FULL_DATASET = False
    SAMPLE_SIZE_FOR_K_SELECTION = 500_000  # Sample 500K for K determination
else:
    print('✓ Dataset size manageable - using full dataset')
    USE_FULL_DATASET = True
    embeddings = load_embeddings_from_files(INPUT_FILES)
    n_samples = embeddings.shape[0]
    print(f'Loaded embeddings shape: {embeddings.shape}')

# For ultra-large datasets, sample from each file proportionally
if not USE_FULL_DATASET:
    print(f'Sampling {SAMPLE_SIZE_FOR_K_SELECTION:,} samples for K-selection...')
    sampled_arrays = []
    
    for f in INPUT_FILES:
        if not f.exists():
            continue
        arr, _, n_rows = load_single_file_smart(f)  # Auto-detect dimension
        # Sample proportionally
        n_sample_from_file = min(n_rows, SAMPLE_SIZE_FOR_K_SELECTION // len(INPUT_FILES))
        if n_sample_from_file > 0:
            idx = np.random.RandomState(RANDOM_STATE).choice(n_rows, n_sample_from_file, replace=False)
            sampled_arrays.append(arr[idx])
    
    embeddings = np.vstack(sampled_arrays)
    n_samples = embeddings.shape[0]
    print(f'Sampled embeddings shape: {embeddings.shape}')
    del sampled_arrays
    gc.collect()

# Sample for silhouette if still too large
if n_samples > SAMPLE_FOR_METRICS:
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(n_samples, SAMPLE_FOR_METRICS, replace=False)
    sample_for_sil = embeddings[sample_idx]
    print(f'Using {SAMPLE_FOR_METRICS:,} samples for silhouette calculation')
else:
    sample_for_sil = embeddings
    print(f'Using all {n_samples:,} samples for silhouette calculation')

# Compute metrics for each K
print(f'\n🔄 Testing K values from {K_RANGE.start} to {K_RANGE.stop-1}...')
for k in tqdm(list(K_RANGE), desc='K-selection', unit='K'):
    start_time = time.time()
    # Use MiniBatchKMeans for speed; use full KMeans for final training
    # n_jobs=-1 to use all CPU cores
    km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, batch_size=4096, n_init=3, max_iter=100)
    km.fit(embeddings)
    inertia.append(km.inertia_)
    # silhouette on sample only
    labels_sample = km.predict(sample_for_sil)
    sil = silhouette_score(sample_for_sil, labels_sample, n_jobs=-1) if len(np.unique(labels_sample))>1 else -1
    sil_scores.append(sil)
    elapsed = time.time() - start_time
    tqdm.write(f'  K={k:2d} → Inertia: {km.inertia_:12,.2f}, Silhouette: {sil:.4f} ({elapsed:.1f}s)')

# Plot results
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(14,4))
ax[0].plot(list(K_RANGE), inertia, '-o')
ax[0].set_xlabel('K')
ax[0].set_ylabel('Inertia')
ax[0].set_title('Elbow: Inertia vs K')
ax[0].grid(True, alpha=0.3)

ax[1].plot(list(K_RANGE), sil_scores, '-o', color='orange')
ax[1].set_xlabel('K')
ax[1].set_ylabel('Silhouette Score (sample)')
ax[1].set_title('Silhouette (sample) vs K')
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Recommendation
best_k_sil = K_RANGE[np.argmax(sil_scores)]

print(f'\n📊 Recommendation: K={best_k_sil} has highest silhouette score ({max(sil_scores):.4f})')
print('👀 Review elbow plot for additional insight')


## Part 3 — Train final K-Means and save model
Pilih K berdasarkan hasil Part 2. Untuk dataset besar gunakan `MiniBatchKMeans` atau `KMeans` dengan `n_init` lebih kecil untuk stabilitas/performance.

In [ ]:
# Part 3 — Fit final model and save
CHOSEN_K = 4  # <-- replace with your chosen K from previous cell
USE_MINIBATCH = True  # set False to use sklearn.KMeans (may be slower)

# For ultra-large datasets, we need incremental fitting
if not USE_FULL_DATASET:
    print('⚠️ Ultra-large dataset: Using incremental fitting per file')
    
    if USE_MINIBATCH:
        model = MiniBatchKMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, batch_size=4096, n_init=10)
    else:
        print('❌ Cannot use standard KMeans on ultra-large dataset - switching to MiniBatch')
        model = MiniBatchKMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, batch_size=4096, n_init=10)
    
    print('Fitting model incrementally per file...')
    for file_idx, f in enumerate(INPUT_FILES, 1):
        if not f.exists():
            print(f'  ⚠️ Skipping missing file: {f.name}')
            continue
            
        print(f'  [{file_idx}/{len(INPUT_FILES)}] Processing {f.name}...')
        arr, _, n_rows = load_single_file_smart(f)  # Auto-detect dimension
        
        # Process in chunks to avoid memory issues
        CHUNK_SIZE = 100_000
        n_chunks = (n_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
        for start in tqdm(range(0, n_rows, CHUNK_SIZE), desc=f'  File {file_idx}', total=n_chunks, unit='chunk'):
            end = min(start + CHUNK_SIZE, n_rows)
            chunk = arr[start:end]
            if isinstance(chunk, np.memmap):
                chunk = np.array(chunk)  # Convert chunk to array for fitting
            model.partial_fit(chunk)
        
        del arr
        gc.collect()
    
    print('✓ Incremental fitting complete')
    
    # Get labels by predicting on each file
    print('Generating labels for all samples...')
    all_labels = []
    for file_idx, f in enumerate(INPUT_FILES, 1):
        if not f.exists():
            continue
            
        print(f'  [{file_idx}/{len(INPUT_FILES)}] Predicting {f.name}...')
        arr, _, n_rows = load_single_file_smart(f)  # Auto-detect dimension
        
        CHUNK_SIZE = 100_000
        file_labels = np.zeros(n_rows, dtype=np.int32)
        n_chunks = (n_rows + CHUNK_SIZE - 1) // CHUNK_SIZE
        for start in tqdm(range(0, n_rows, CHUNK_SIZE), desc=f'  Predict {file_idx}', total=n_chunks, unit='chunk'):
            end = min(start + CHUNK_SIZE, n_rows)
            chunk = arr[start:end]
            if isinstance(chunk, np.memmap):
                chunk = np.array(chunk)
            file_labels[start:end] = model.predict(chunk)
        
        all_labels.append(file_labels)
        del arr
        gc.collect()
    
    labels = np.concatenate(all_labels)
    centroids = model.cluster_centers_
    
else:
    # Standard approach for manageable datasets
    if USE_MINIBATCH:
        model = MiniBatchKMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, batch_size=4096, n_init=10)
        print('Fitting MiniBatchKMeans on full dataset...')
    else:
        # Use standard KMeans (auto uses all CPU cores via OpenMP)
        model = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, n_init=10)
        print('Fitting standard KMeans on full dataset...')
    
    start_time = time.time()
    model.fit(embeddings)
    elapsed = time.time() - start_time
    print(f'✓ Fitting completed in {elapsed/60:.1f} minutes')
    labels = model.labels_
    centroids = model.cluster_centers_

# Save model and labels
out_model = Path('model_kmeans_log.pkl')
joblib.dump(model, out_model)
np.save('cluster_labels.npy', labels)
np.save('cluster_centroids.npy', centroids)

print(f'✓ Saved model: {out_model}')
print(f'✓ Saved labels: cluster_labels.npy ({len(labels):,} samples)')
print(f'✓ Saved centroids: cluster_centroids.npy ({centroids.shape})')


## Part 4 — Cluster analysis & sampling
Tunjukkan ukuran cluster, beberapa contoh log per cluster (jika mapping ke log asli tersedia), dan simpan ringkasan ke CSV untuk analisa lebih lanjut.

In [ ]:
# Part 4 — Analyze clusters
from collections import Counter
counts = Counter(labels)
print('Cluster sizes:')
for k, c in sorted(counts.items()):
    print(f' - Cluster {k}: {c} samples')

# If you have original logs aligned to embeddings, load mapping here and sample examples.
# Example: assume a CSV with original logs in same order as embeddings: logs.csv with column 'log'
logs_csv = Path('../dataset/logs_in_order.csv')  # optional: provide a file mapping
if logs_csv.exists():
    df_logs = pd.read_csv(logs_csv)
    df_logs['cluster'] = labels
    # save cluster summary
    summary_path = Path('cluster_summary.csv')
    df_logs.to_csv(summary_path, index=False)
    print('Saved cluster summary to', summary_path)
else:
    print('No original logs mapping found at', logs_csv, '. Showing indices for samples instead.')
    for k in range(CHOSEN_K):
        idxs = np.where(labels==k)[0][:5]
        print(f' Cluster {k} sample indices: {list(idxs)}')

## Part 5 — Compare variants (optional)
Jika Anda ingin membandingkan Baseline vs PCA256 performance, jalankan eksperimen kecil: hitung silhouette pada subset sama untuk kedua variant dan bandingkan.

In [ ]:
# Part 5 — Quick comparison function (Baseline vs PCA variant)
def compare_variants(baseline_files, variant_files, sample_size=50000):
    """
    Compare clustering quality between two variants (e.g., baseline vs PCA)
    
    Args:
        baseline_files: list of Path objects for baseline embeddings
        variant_files: list of Path objects for variant embeddings
        sample_size: number of samples to use for comparison
    
    Returns:
        (baseline_silhouette, variant_silhouette)
    """
    # load sample from baseline and variant (matching random indices)
    base = load_embeddings_from_files(baseline_files)
    var = load_embeddings_from_files(variant_files)
    n = min(base.shape[0], var.shape[0], sample_size)
    idx = np.random.RandomState(RANDOM_STATE).choice(base.shape[0], n, replace=False)
    base_s = base[idx]
    var_s = var[idx]
    # cluster with same K (small K for speed)
    k = 10
    km_base = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE).fit(base_s)
    km_var = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE).fit(var_s)
    sil_base = silhouette_score(base_s, km_base.labels_)
    sil_var = silhouette_score(var_s, km_var.labels_)
    return sil_base, sil_var

# Example usage:
# Compare BGL baseline vs PCA256
# baseline_files = [Path('/media/.../dataset_vector/after_preprocessed_bgl_embeddings.npy')]
# pca_files = [Path('/media/.../dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy')]
# sil_base, sil_pca = compare_variants(baseline_files, pca_files, sample_size=20000)
# print(f'Baseline silhouette: {sil_base:.4f}')
# print(f'PCA256 silhouette: {sil_pca:.4f}')
# print(f'Quality loss: {((sil_base - sil_pca) / sil_base * 100):.2f}%')


## Part 6 — Configuration Examples & Recommended Settings

### **Quick Start: How to Use This Notebook**

Cukup edit **Cell 2** dan ganti `INPUT_FILES` sesuai kebutuhan:

```python
# Example 1: BGL only (baseline, 12GB)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
]

# Example 2: Thunderbird only (PCA128, 100GB - recommended!)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
]

# Example 3: Both combined (PCA256, 204GB)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_thunderbird_pca256_embeddings.npy"),
]
```

### **Dataset Handling by Size:**

| Dataset | File Path | Size | Strategy | Time |
|---------|-----------|------|----------|------|
| **BGL baseline** | `.../after_preprocessed_bgl_embeddings.npy` | 12 GB | Load to RAM | 15 min |
| **BGL PCA256** | `.../after_preprocessed_bgl_pca256_embeddings.npy` | 4 GB | Load to RAM | 8 min |
| **Thunderbird baseline** | `.../after_preprocessed_thunderbird_embeddings.npy` | 600 GB | Incremental | 5 hours |
| **Thunderbird PCA128** | `.../after_preprocessed_thunderbird_pca128_embeddings.npy` | 100 GB | Incremental | 3 hours |

### **Recommended Configurations:**

#### **1. BGL Only (Fast Prototyping)**
```python
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
]
SAMPLE_FOR_METRICS = 4_000_000  # Use all
USE_MINIBATCH = False  # Full KMeans
CHOSEN_K = 4  # Adjust based on results
```
**Time:** 15-20 minutes | **Quality:** ⭐⭐⭐⭐⭐

---

#### **2. Thunderbird Only (Production)**
```python
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
]
SAMPLE_FOR_METRICS = 200_000  # Sample
USE_MINIBATCH = True  # Required
CHOSEN_K = 6  # Larger dataset may need more clusters
```
**Time:** 2-3 hours | **Quality:** ⭐⭐⭐⭐

---

#### **3. Both Combined (Comprehensive)**
```python
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_thunderbird_pca256_embeddings.npy"),
]
SAMPLE_FOR_METRICS = 500_000
USE_MINIBATCH = True
CHOSEN_K = 8  # Combined may have more diverse patterns
```
**Time:** 4-5 hours | **Quality:** ⭐⭐⭐⭐

---

### **File Path Pattern Reference:**

Baseline (768-dim):
```
/media/bioinfo04/Expansion/2427051003_dataset_vector/
  ├── after_preprocessed_bgl_embeddings.npy        (12 GB)
  └── after_preprocessed_thunderbird_embeddings.npy (600 GB)
```

Normalized (768-dim):
```
/media/bioinfo04/Expansion/2427051003_dataset_vector_normalized/
  ├── after_preprocessed_bgl_normalized_embeddings.npy
  └── after_preprocessed_thunderbird_normalized_embeddings.npy
```

PCA256 (256-dim):
```
/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/
  ├── after_preprocessed_bgl_pca256_embeddings.npy        (4 GB)
  └── after_preprocessed_thunderbird_pca256_embeddings.npy (200 GB)
```

PCA128 (128-dim):
```
/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/
  ├── after_preprocessed_bgl_pca128_embeddings.npy        (2 GB)
  └── after_preprocessed_thunderbird_pca128_embeddings.npy (100 GB)
```

---

### **Model Persistence & Inference:**

Setelah training selesai, gunakan model untuk inference:

```python
# Load trained model
model = joblib.load('model_kmeans_log.pkl')
pca_model = joblib.load('/path/to/pca_model_256.pkl')  # if using PCA

# Inference on new logs
new_embeddings = bert_model.encode(new_logs)  # Get BERT embeddings
new_embeddings_pca = pca_model.transform(new_embeddings)  # Apply PCA if needed
cluster_ids = model.predict(new_embeddings_pca)  # Predict cluster
```

---

### **Evaluation Metrics:**
- `silhouette_score`: Cluster separation quality (higher = better)
- `davies_bouldin_score`: Cluster compactness (lower = better)
- `calinski_harabasz_score`: Variance ratio (higher = better)
- Manual inspection: Sample 10-50 logs per cluster untuk validasi semantik

## Part 7 — KMeans vs MiniBatchKMeans: Direct Comparison

**Tujuan:** Membandingkan kualitas clustering antara standard KMeans dan MiniBatchKMeans dengan parameter identik.

**Metode:**
- Sample dataset yang sama untuk kedua model
- Parameter identik: `n_clusters`, `random_state`, `n_init`
- Ukur: **inertia** (lower = tighter clusters) dan **silhouette score** (higher = better separation)
- Waktu training untuk masing-masing model

**Expected Results:**
- KMeans: Lebih akurat, training lebih lama
- MiniBatchKMeans: Sedikit kurang akurat, training jauh lebih cepat

**Catatan:** Untuk dataset BGL (12GB), perbedaan kualitas biasanya **< 5%**, sedangkan speedup bisa **3-10x** lebih cepat.

In [ ]:
# Part 7 — Direct comparison: KMeans vs MiniBatchKMeans
# =====================================================================
# This cell compares clustering quality between standard KMeans and MiniBatchKMeans
# on the same dataset with identical parameters
# =====================================================================

import time
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# Configuration
COMPARISON_SAMPLE_SIZE = 100_000  # Use smaller sample for fair speed comparison
COMPARISON_K = 4  # Number of clusters to test
N_INIT = 10  # Number of initializations
BATCH_SIZE = 4096  # For MiniBatchKMeans
MAX_ITER = 300  # Max iterations

print("="*70)
print("KMEANS vs MINIBATCHKMEANS COMPARISON")
print("="*70)
print(f"Sample size: {COMPARISON_SAMPLE_SIZE:,}")
print(f"K clusters: {COMPARISON_K}")
print(f"n_init: {N_INIT}")
print(f"Random state: {RANDOM_STATE}")
print(f"MiniBatch batch_size: {BATCH_SIZE}")
print("="*70 + "\n")

# Step 1: Prepare sample data
print("📊 Preparing sample data...")
try:
    # Try to use embeddings from previous cells if available
    if 'embeddings' in locals() or 'embeddings' in globals():
        print("  ✓ Using embeddings from previous cells")
        if embeddings.shape[0] > COMPARISON_SAMPLE_SIZE:
            sample_idx = np.random.RandomState(RANDOM_STATE).choice(
                embeddings.shape[0], COMPARISON_SAMPLE_SIZE, replace=False
            )
            sample_data = embeddings[sample_idx]
        else:
            sample_data = embeddings
        print(f"  ✓ Sample shape: {sample_data.shape}")
    else:
        # Load fresh sample from files
        print("  ⚠️ No embeddings found, loading from files...")
        sampled_arrays = []
        for f in INPUT_FILES:
            if not f.exists():
                continue
            arr, _, n_rows = load_single_file_smart(f, embedding_dim=768)
            n_sample = min(n_rows, COMPARISON_SAMPLE_SIZE // len(INPUT_FILES))
            if n_sample > 0:
                idx = np.random.RandomState(RANDOM_STATE).choice(n_rows, n_sample, replace=False)
                sampled_arrays.append(arr[idx])
        sample_data = np.vstack(sampled_arrays)
        print(f"  ✓ Loaded sample shape: {sample_data.shape}")
        del sampled_arrays
        gc.collect()
except Exception as e:
    print(f"  ❌ Error loading data: {e}")
    print("  → Please run previous cells first to load embeddings")
    raise

# Ensure we have manageable sample
actual_sample_size = min(sample_data.shape[0], COMPARISON_SAMPLE_SIZE)
if sample_data.shape[0] > COMPARISON_SAMPLE_SIZE:
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(
        sample_data.shape[0], COMPARISON_SAMPLE_SIZE, replace=False
    )
    sample_data = sample_data[sample_idx]

print(f"\n✓ Final sample size: {sample_data.shape[0]:,} samples\n")

# Step 2: Train standard KMeans
print("🔵 Training standard KMeans...")
print(f"   Parameters: K={COMPARISON_K}, n_init={N_INIT}, max_iter={MAX_ITER}")
start_time = time.time()
kmeans_standard = KMeans(
    n_clusters=COMPARISON_K,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
    max_iter=MAX_ITER,
    verbose=0
)
kmeans_standard.fit(sample_data)
kmeans_time = time.time() - start_time
kmeans_labels = kmeans_standard.labels_
kmeans_inertia = kmeans_standard.inertia_
kmeans_silhouette = silhouette_score(sample_data, kmeans_labels, sample_size=min(50000, len(sample_data)))
kmeans_iterations = kmeans_standard.n_iter_

print(f"   ✓ Training completed in {kmeans_time:.2f}s ({kmeans_time/60:.2f} min)")
print(f"   ✓ Converged in {kmeans_iterations} iterations")
print(f"   ✓ Inertia: {kmeans_inertia:,.2f}")
print(f"   ✓ Silhouette: {kmeans_silhouette:.4f}")

# Step 3: Train MiniBatchKMeans
print(f"\n🟠 Training MiniBatchKMeans...")
print(f"   Parameters: K={COMPARISON_K}, n_init={N_INIT}, batch_size={BATCH_SIZE}, max_iter={MAX_ITER}")
start_time = time.time()
kmeans_minibatch = MiniBatchKMeans(
    n_clusters=COMPARISON_K,
    random_state=RANDOM_STATE,
    n_init=N_INIT,
    batch_size=BATCH_SIZE,
    max_iter=MAX_ITER,
    verbose=0
)
kmeans_minibatch.fit(sample_data)
minibatch_time = time.time() - start_time
minibatch_labels = kmeans_minibatch.labels_
minibatch_inertia = kmeans_minibatch.inertia_
minibatch_silhouette = silhouette_score(sample_data, minibatch_labels, sample_size=min(50000, len(sample_data)))
minibatch_iterations = kmeans_minibatch.n_iter_

print(f"   ✓ Training completed in {minibatch_time:.2f}s ({minibatch_time/60:.2f} min)")
print(f"   ✓ Converged in {minibatch_iterations} iterations")
print(f"   ✓ Inertia: {minibatch_inertia:,.2f}")
print(f"   ✓ Silhouette: {minibatch_silhouette:.4f}")

# Step 4: Calculate differences
print("\n" + "="*70)
print("COMPARISON RESULTS")
print("="*70)

speedup = kmeans_time / minibatch_time
inertia_diff_pct = ((minibatch_inertia - kmeans_inertia) / kmeans_inertia) * 100
silhouette_diff_pct = ((minibatch_silhouette - kmeans_silhouette) / kmeans_silhouette) * 100

results_df = pd.DataFrame({
    'Metric': ['Training Time (s)', 'Training Time (min)', 'Iterations', 'Inertia', 'Silhouette Score'],
    'KMeans': [
        f'{kmeans_time:.2f}',
        f'{kmeans_time/60:.2f}',
        kmeans_iterations,
        f'{kmeans_inertia:,.2f}',
        f'{kmeans_silhouette:.4f}'
    ],
    'MiniBatchKMeans': [
        f'{minibatch_time:.2f}',
        f'{minibatch_time/60:.2f}',
        minibatch_iterations,
        f'{minibatch_inertia:,.2f}',
        f'{minibatch_silhouette:.4f}'
    ],
    'Difference': [
        f'{speedup:.2f}x faster',
        f'{speedup:.2f}x faster',
        f'{minibatch_iterations - kmeans_iterations:+d}',
        f'{inertia_diff_pct:+.2f}%',
        f'{silhouette_diff_pct:+.2f}%'
    ]
})

print(results_df.to_string(index=False))
print("="*70)

# Step 5: Summary & Interpretation
print("\n📈 INTERPRETATION:")
print(f"   Speed: MiniBatchKMeans is {speedup:.1f}x faster")
print(f"   Quality Loss:")
print(f"      - Inertia: {abs(inertia_diff_pct):.2f}% {'worse' if inertia_diff_pct > 0 else 'better'}")
print(f"      - Silhouette: {abs(silhouette_diff_pct):.2f}% {'worse' if silhouette_diff_pct < 0 else 'better'}")

if abs(silhouette_diff_pct) < 5:
    quality_verdict = "✅ MINIMAL QUALITY LOSS (<5%)"
    recommendation = "MiniBatchKMeans is excellent choice for this dataset"
elif abs(silhouette_diff_pct) < 10:
    quality_verdict = "⚠️  ACCEPTABLE QUALITY LOSS (5-10%)"
    recommendation = "MiniBatchKMeans is good trade-off for large datasets"
else:
    quality_verdict = "❌ SIGNIFICANT QUALITY LOSS (>10%)"
    recommendation = "Consider using standard KMeans or tune MiniBatch parameters"

print(f"\n   {quality_verdict}")
print(f"   💡 Recommendation: {recommendation}")

# Step 6: Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Training time comparison
methods = ['KMeans', 'MiniBatch']
times = [kmeans_time, minibatch_time]
colors = ['#3498db', '#e67e22']
axes[0].bar(methods, times, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Training Time (seconds)')
axes[0].set_title('Training Speed Comparison')
axes[0].grid(axis='y', alpha=0.3)
for i, (m, t) in enumerate(zip(methods, times)):
    axes[0].text(i, t + max(times)*0.02, f'{t:.2f}s', ha='center', fontweight='bold')

# Plot 2: Inertia comparison
inertias = [kmeans_inertia, minibatch_inertia]
axes[1].bar(methods, inertias, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Inertia (lower = better)')
axes[1].set_title('Inertia Comparison')
axes[1].grid(axis='y', alpha=0.3)
axes[1].ticklabel_format(style='plain', axis='y')
for i, (m, iner) in enumerate(zip(methods, inertias)):
    axes[1].text(i, iner + max(inertias)*0.01, f'{iner:,.0f}', ha='center', fontweight='bold', fontsize=9)

# Plot 3: Silhouette score comparison
silhouettes = [kmeans_silhouette, minibatch_silhouette]
axes[2].bar(methods, silhouettes, color=colors, alpha=0.7, edgecolor='black')
axes[2].set_ylabel('Silhouette Score (higher = better)')
axes[2].set_title('Silhouette Score Comparison')
axes[2].set_ylim([min(silhouettes)*0.95, max(silhouettes)*1.05])
axes[2].grid(axis='y', alpha=0.3)
for i, (m, sil) in enumerate(zip(methods, silhouettes)):
    axes[2].text(i, sil + (max(silhouettes)-min(silhouettes))*0.1, f'{sil:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("💾 SUMMARY FOR THESIS:")
print("="*70)
print(f"Dataset: {sample_data.shape[0]:,} samples, {sample_data.shape[1]} dimensions")
print(f"Clusters (K): {COMPARISON_K}")
print(f"")
print(f"Standard KMeans:")
print(f"  - Time: {kmeans_time:.2f}s ({kmeans_time/60:.2f} min)")
print(f"  - Inertia: {kmeans_inertia:,.2f}")
print(f"  - Silhouette: {kmeans_silhouette:.4f}")
print(f"")
print(f"MiniBatch KMeans (batch_size={BATCH_SIZE}):")
print(f"  - Time: {minibatch_time:.2f}s ({minibatch_time/60:.2f} min)")
print(f"  - Inertia: {minibatch_inertia:,.2f}")
print(f"  - Silhouette: {minibatch_silhouette:.4f}")
print(f"")
print(f"Trade-off Analysis:")
print(f"  - Speedup: {speedup:.2f}x faster")
print(f"  - Quality loss: {abs(silhouette_diff_pct):.2f}%")
print(f"  - Verdict: {quality_verdict}")
print("="*70)